# Practice Exam — Form A · CCAR-F simulacrum

**Original practice items written for this study kit — NOT real exam content.** They reproduce the
*style and cognitive level* of the live exam (scenario-based, judgment-not-recall), not any actual
question. Companion to [`MAPPING.md`](../MAPPING.md) and the retired-practice-exam replacement noted in
[`STUDY_PLAN.md`](../STUDY_PLAN.md).

## What this is
- **33 questions across all 6 scenarios.** The real exam presents **4 of 6** scenarios (~60 items); this
  form covers all six for full-surface practice.
- **Mixed format**, matching the current guide: most items are single-answer; several are
  **multiple-response** — marked **_(Select 2.)_**. On a Select-2 item there is **no partial credit** —
  you need *both* correct options and *no* wrong ones. Evaluate every option as an independent
  true/false, don't just pick "the best."

## How to use
1. **Run the clock cell** (just below). It starts the timer and stores it under `personal/`,
   so it survives a kernel restart: if the kernel dies, you don't lose the attempt.
2. **One question, one cell.** Every question has its own answer cell underneath it. Type the
   letter between the quotes and run it. One or several, whichever is faster for you:
   `"B"`, `"B, D"`, `"BD"`. Each run reports the clock and your pace.
3. **Reveal nothing until the end.** Answers live in hidden `<!-- -->` comments below each
   question. `grade()` shows the rationale for the ones you *missed* only.
4. **No notes, no docs, no Claude.** If you don't know, mark something anyway — there is no
   guessing penalty.
5. **When you're done, hit "Run All"** (the ▶▶ button at the top of the notebook). It replays
   every answer cell and then grades you. Do *not* Run All before you start — that would grade
   an empty attempt.

## Pace and scoring
- **Limit: 66 minutes** (33 questions x 2 min, the real exam's pace of 120 min / 60 questions).
  Finishing near 45 min means you're comfortable. The clock warns you when you fall behind.
- **Target: >= 80%** (~27/33) before sitting the real exam. Below that, the `Tag:` lines on your
  misses tell you which notebook to revisit.
- On **Select 2** items there is **no partial credit**: you need both correct options and no extras.

---


In [ ]:
# ⏱ CLOCK — run this ONCE to start the attempt.
# (If the kernel restarts, run it again: it resumes, it does not reset.)
import examkit as ex

ex.start()

# Wipe the previous attempt and begin from scratch:  ex.start(restart=True)
# Check the clock at any point:                      ex.clock()


## Scenario 1 — Customer Support Resolution Agent

_Primary domains: D1 Agentic Architecture & Orchestration · D2 Tool Design & MCP Integration · D5 Context Management & Reliability._


---

**Q1.** Your support agent calls `process_refund` autonomously. Policy states that any refund above $500 must be approved by a human manager before funds move, but in the last audit 7 refunds between $600 and $1,200 were issued because the model, under a persuasive customer message, judged them "clearly justified." The prompt already says "NEVER auto-approve refunds over $500 — always escalate." What is the most reliable fix?  _(Select 1.)_

- **A.** Split `process_refund` into `process_small_refund` and `process_large_refund` and only register the small one in the toolset.
- **B.** Lower the model temperature to 0 so refund decisions become deterministic and stop being swayed by customer wording.
- **C.** Add a `PreToolUse` hook on `process_refund` that inspects the `amount` argument and blocks (deny) the call when it exceeds $500, forcing the escalation path.
- **D.** Strengthen the system prompt with capitalized warnings and two few-shot examples of correctly refusing a $700 refund.
<!--ANSWER
Correct: C
Tag: D1.5 · Code must, prompt should
Why: A hard dollar ceiling is a guaranteed requirement, so it belongs in deterministic code — a `PreToolUse` hook that reads the tool arguments and denies over-threshold calls enforces it regardless of how persuasive the prompt input is. D only makes the prompt louder; the same failure already happened despite an explicit instruction. B reduces variance but a temperature-0 model can still confidently decide to approve a $700 refund. A looks structural but the model could still call the large-refund tool if it were ever registered, and here the agent legitimately needs to handle large refunds — by escalating, not by losing the capability; the hook gates the action without removing it.
-->


In [ ]:
ex.answer(Q1="")


---

**Q2.** Customers report the agent replying "I couldn't find any orders on your account, so there's nothing to refund" — but many of them do have orders. Investigation shows `lookup_order` sometimes returns `[]` because the orders database read-replica timed out, and other times returns `[]` because the verified customer genuinely has no orders. Both cases surface to the model identically. Which change best fixes the wrong resolutions?  _(Select 1.)_

- **A.** Add a retry wrapper that calls `lookup_order` up to three times before returning any result to the model.
- **B.** Have `lookup_order` return a structured payload distinguishing a valid empty result (`status: "ok", orders: []`) from an access failure (`status: "error", errorCategory: "upstream_timeout", isRetryable: true`) so the model treats the two differently.
- **C.** Cache the last successful `lookup_order` response per customer and serve it whenever the live call returns an empty list.
- **D.** Change the prompt to instruct the model to always assume orders exist and ask the customer to re-verify when none are returned.
<!--ANSWER
Correct: B
Tag: D2.2 · Structured errors over generic (access-fail vs valid-empty)
Why: The root cause is that a database failure and a genuine no-orders result are indistinguishable to the model, so it narrates "no orders" for both. Returning a structured error that carries `errorCategory` and `isRetryable` lets the agent retry/escalate on a timeout while correctly telling a truly order-less customer the truth. A blindly retries even the valid-empty case and still can't tell the model which case it saw. D makes the agent hallucinate orders and annoy customers who really have none. C can serve stale or wrong data and still masks the failure signal.
-->


In [ ]:
ex.answer(Q2="")


---

**Q3.** A billing-dispute conversation runs 40+ turns. Early on the customer stated the disputed charge ID (`CHG-88231`), the amount ($149.99), and that they want a partial credit, not a full reversal. By turn 35 the agent, working from a rolling conversation summary, offers a full reversal on the wrong charge. Token usage shows the summary is being recompacted every ~10 turns. What is the correct architectural fix?  _(Select 1.)_

- **A.** Raise the summarization frequency to every 3 turns so details are refreshed more often.
- **B.** Increase the context window model tier so the full transcript fits without any summarization.
- **C.** Persist the confirmed case facts (charge ID, amount, requested remedy) in a dedicated structured block kept outside the rolling summary, and re-inject it verbatim on every turn.
- **D.** Instruct the model in the system prompt to "always remember the original charge ID and amount for the entire conversation."
<!--ANSWER
Correct: C
Tag: D5.1 · Context hygiene — case facts outside the rolling summary
Why: Transactional facts that must survive verbatim should live in a durable, structured block that is never subject to lossy recompaction — re-injecting them each turn guarantees the agent acts on the right charge and remedy. B just delays the same lossy-summary and lost-in-the-middle problem at higher cost and doesn't stop details from decaying. A summarizes more often, which can degrade the facts faster, not preserve them. D relies on the prompt to protect data the summarization step keeps overwriting — the exact failure observed.
-->


In [ ]:
ex.answer(Q3="")


---

**Q4.** The agent is meant to escalate via `escalate_to_human` when it should, but reviewers find both over- and under-escalation. Which signals are RELIABLE, principled triggers to route a case to a human? _(Select 2.)_

- **A.** The model's self-reported confidence for its proposed resolution drops below 0.6.
- **B.** The customer explicitly asks to speak to a person or invokes a right (e.g., "I want to file a formal complaint").
- **C.** The conversation has exceeded eight turns regardless of whether progress is being made.
- **D.** The request falls into a gap not covered by any policy the agent was given (e.g., a scenario the refund rules don't address).
- **E.** The customer's message sentiment is classified as "angry" or "frustrated."
<!--ANSWER
Correct: B, D
Tag: D5.2 · Escalation triggers (customer ask / policy gap; sentiment & self-confidence unreliable)
Why: A direct customer request and a genuine policy gap are objective, principled escalation triggers — the agent cannot legitimately resolve what the customer won't let it or what no policy covers. A is unreliable: self-reported confidence is poorly calibrated and easily miscalibrated by phrasing. E uses sentiment as a proxy — an angry customer may still be fully resolvable, and a calm one may need a human; it drives both over- and under-escalation. C fixates on raw turn count; the real signal is lack of *progress*, not elapsed turns.
-->


In [ ]:
ex.answer(Q4="")   # Select 2 — TWO letters


---

**Q5.** Your team is adding a "close ticket" handoff: when the agent believes a case is resolved it should transfer to a `finalize_resolution` step that writes the outcome to the CRM. You want to guarantee no ticket is finalized unless (a) the customer was identified with a verified ID and (b) either a refund was processed or the customer explicitly confirmed satisfaction. Which measures correctly enforce this handoff gate? _(Select 2.)_

- **A.** A system-prompt instruction telling the model to only finalize after verifying identity and confirming resolution.
- **B.** Adding a few-shot example to the prompt showing the agent declining to finalize an unverified case.
- **C.** A programmatic precondition check on the `finalize_resolution` transition that reads state flags (`verified_customer_id` present AND (`refund_processed` OR `customer_confirmed`)) and refuses the handoff otherwise.
- **D.** A `PreToolUse` hook on `finalize_resolution` that denies the call when the required state flags are absent, returning a message that names the missing precondition.
- **E.** A post-hoc nightly CRM audit job that flags tickets finalized without a verified customer ID for manual review.
<!--ANSWER
Correct: C, D
Tag: D1.4 · Enforcement/handoff gates (guaranteed requirement -> deterministic gate)
Why: A guaranteed handoff precondition must be enforced in code, either as a precondition check on the state transition (C) or as a deny-capable `PreToolUse` hook that inspects state and blocks the call while naming the missing precondition (D) — both make the invalid handoff impossible. A and B are prompt-level and can be bypassed by any single mis-judgment, exactly what a guaranteed requirement can't tolerate. E catches violations only after they've been written to the CRM; detection is not prevention.
-->


In [ ]:
ex.answer(Q5="")   # Select 2 — TWO letters


---

**Q6.** `get_customer` returns phone numbers in whatever format the source system stored them (`+1 (415) 555-0134`, `415.555.0134`, `4155550134`). Downstream, `escalate_to_human` requires E.164 (`+14155550134`) and rejects anything else, causing intermittent escalation failures. You want every phone number normalized before the model ever reasons about it, without trusting the model to reformat correctly. What is the best approach?  _(Select 1.)_

- **A.** Add validation inside `escalate_to_human` that rejects non-E.164 input with an error so the model retries with a corrected format.
- **B.** Add a `PostToolUse` hook on `get_customer` that rewrites the `phone` field to E.164 in the tool result before it is returned to the model.
- **C.** Expand the `get_customer` tool description to explain the three possible phone formats the model may encounter.
- **D.** Instruct the model in the system prompt to always convert phone numbers to E.164 before calling `escalate_to_human`.
<!--ANSWER
Correct: B
Tag: D1.5 · PostToolUse normalization (deterministic transform, not model reformatting)
Why: Normalization is a deterministic string transform, so a `PostToolUse` hook that rewrites the tool output guarantees the model always sees canonical E.164 and never has to reformat — the failure is designed out. D trusts the model to reformat every time, which is exactly the unreliable step causing intermittent failures. A only rejects after the fact and forces retry loops driven by the same unreliable model reformatting. C gives the model more to reason about but still leaves the error-prone conversion in the model's hands.
-->


In [ ]:
ex.answer(Q6="")


## Scenario 2 — Code Generation with Claude Code

_Primary domains: D3 Claude Code Configuration & Workflows · D5 Context Management & Reliability._


---

**Q7.** A monorepo's root `CLAUDE.md` has grown to ~700 lines: shared build/test commands, a long "API authentication conventions" section that only the `services/auth` subtree cares about, and one engineer's personal git aliases. Teammates report that Claude frequently ignores the auth conventions on long sessions, and the personal aliases keep showing up in others' checkouts. What is the best restructuring?  _(Select 1.)_

- **A.** Keep everything in the root `CLAUDE.md` but prepend "IMPORTANT: always obey the auth conventions" so the model stops dropping them.
- **B.** Move the entire root file to `~/.claude/CLAUDE.md` so it is loaded globally and never truncated per-project.
- **C.** Keep shared build/test commands in the root file, move the auth conventions into `services/auth/CLAUDE.md` (or a `.claude/rules/` topic file `@import`ed where relevant), and move the personal aliases to `~/.claude/CLAUDE.md`.
- **D.** Leave the file as-is and switch Claude to a larger model so the full 700 lines stay reliably in attention.
<!--ANSWER
Correct: C
Tag: D3.1 · CLAUDE.md hierarchy, scope, and modularity
Why: The fix is structural, not a bigger model (D) or a louder prompt (A). Subtree-specific conventions belong where they apply (directory-level or a modular `@import`/rules file), so they load in context precisely when relevant; personal, non-shared config belongs at user level (`~/.claude`), which is not version-controlled. B is wrong because user-level config would stop sharing the build commands the whole team needs.
-->


In [ ]:
ex.answer(Q7="")


---

**Q8.** A shared project slash command `.claude/commands/release-notes.md` shells out to `git log` and diff summaries; its raw output floods the main conversation and crowds out the code being worked on, and users must hand-edit the command each run to target a specific version tag. Which two changes best address both problems?  _(Select 2.)_

- **A.** Reimplement it as a Skill whose `SKILL.md` frontmatter sets `context: fork` so its verbose discovery runs in an isolated context.
- **B.** Add an `argument-hint` (e.g. `<version-tag>`) so the version is passed as an argument instead of editing the file each run.
- **C.** Move the command from `.claude/commands/` to `~/.claude/commands/` so its output no longer counts against the project.
- **D.** Add `allowed-tools` restricting it to read-only git tools, which will prevent the output from entering the conversation.
- **E.** Switch the underlying model to one with a larger context window so the verbose log no longer matters.
<!--ANSWER
Correct: A, B
Tag: D3.2 · Slash commands vs Skills (context: fork, argument-hint)
Why: `context: fork` isolates verbose/exploratory output so it doesn't pollute the main thread, and `argument-hint` parameterizes the version so no per-run editing is needed. C changes only where it's stored (and un-shares it), not the context flooding. `allowed-tools` (D) restricts which tools run, not whether their output lands in context. A bigger window (E) is over-engineering that ignores the real fix.
-->


In [ ]:
ex.answer(Q8="")   # Select 2 — TWO letters


---

**Q9.** Across a services monorepo, every database migration file — found in `db/migrations/`, `services/*/migrations/`, and a few loose `*.sql` scripts — must include a `-- ROLLBACK:` section and a ticket reference. Claude keeps generating migrations without them. The rule must apply to these scattered files regardless of folder, and to nothing else. What is the most maintainable configuration?  _(Select 1.)_

- **A.** Add a `.claude/rules/migrations.md` file with YAML frontmatter `paths:` globs like `**/migrations/**` and `**/*.sql` describing the rollback/ticket requirement.
- **B.** Drop a `CLAUDE.md` into every current `migrations/` directory restating the requirement.
- **C.** Create a `/migration` slash command that writes the boilerplate and require everyone to invoke it.
- **D.** Add the requirement as a top-level bullet in the root `CLAUDE.md` so it is always loaded.
<!--ANSWER
Correct: A
Tag: D3.3 · Path-specific rules with glob frontmatter
Why: A `paths:` glob rule targets files by pattern wherever they live (including future services and loose `*.sql`), and stays scoped so it never fires on unrelated files. Per-directory `CLAUDE.md` (B) misses the loose scripts and every new subtree. A global root bullet (D) applies everywhere and dilutes context. A slash command (C) is opt-in and doesn't enforce anything when the model writes migrations directly.
-->


In [ ]:
ex.answer(Q9="")


---

**Q10.** An engineer has two tasks queued in Claude Code. Task 1: bump a timeout constant from `30` to `60` in one clearly named config file. Task 2: introduce request-level caching to an API layer — there are several viable placements (middleware, per-handler, a shared decorator), it touches ~15 files, and the wrong choice is expensive to unwind. They currently run both with direct execution and Task 2 keeps producing tangled diffs. Which approach fits each task?  _(Select 1.)_

- **A.** Use direct execution for Task 1 and plan mode for Task 2.
- **B.** Use plan mode for both, since planning never hurts correctness.
- **C.** Use plan mode for Task 1 and direct execution for Task 2, to reserve the fast path for the risky change.
- **D.** Use direct execution for both; add a stricter CLAUDE.md rule to keep Task 2's diffs clean.
<!--ANSWER
Correct: A
Tag: D3.4 · Plan mode vs direct execution
Why: Plan mode earns its overhead when work is multi-file with multiple valid approaches and costly missteps — exactly Task 2. A single, well-scoped, unambiguous edit (Task 1) is faster with direct execution. B is over-applying plan mode; C inverts the fit; D treats an approach-selection problem as a formatting rule.
-->


In [ ]:
ex.answer(Q10="")


---

**Q11.** While debugging an intermittent failure in a large unfamiliar repo, an engineer's session degrades over an hour: Claude runs wide `grep`/file-reads that fill the window, and by the time it has located the bug it has forgotten the reproduction steps and the three suspect files it already ruled out. What is the most effective way to keep the investigation on track?  _(Select 1.)_

- **A.** Run `/compact` repeatedly whenever the window fills, so the whole history is continuously summarized.
- **B.** Keep everything in one conversation but ask Claude to periodically restate all findings so nothing is lost.
- **C.** Switch to a larger-context model and paste the entire repository in at the start so nothing has to be searched.
- **D.** Delegate the wide code discovery to an Explore subagent so its verbose reads stay out of the main context, and record the repro steps and ruled-out files in a scratchpad file the main thread re-reads.
<!--ANSWER
Correct: D
Tag: D5.4 · Large-codebase context management (Explore subagent, scratchpad)
Why: An Explore subagent isolates the token-heavy discovery so it never bloats the main thread, and a scratchpad durably holds case facts (repro steps, ruled-out files) that summaries tend to drop. Blind repeated `/compact` (A) can compress away the very facts being lost. Pasting the whole repo (C) is brute-force over-engineering. Restating everything inline (B) consumes the context it's trying to preserve.
-->


In [ ]:
ex.answer(Q11="")


## Scenario 3 — Multi-Agent Research System

_Primary domains: D1 Agentic Architecture & Orchestration · D2 Tool Design & MCP Integration · D5 Context Management & Reliability._


---

**Q12.** A research system's coordinator delegates to search, document-analysis, synthesis, and report subagents. Tasked with "assess the state of electric-vehicle adoption in Europe," the coordinator spawns four subtasks — all variations on battery chemistry and range. Each subagent executes flawlessly and returns well-cited material, but the final report never mentions charging infrastructure, purchase-subsidy policy, or consumer total-cost-of-ownership. Where is the defect?  _(Select 1.)_

- **A.** The search subagent used weak queries; give it a larger model so it surfaces infrastructure and policy sources on its own.
- **B.** The coordinator's decomposition is too narrow — it framed an inherently multi-facet question as a battery-only problem, so whole subtopics were never assigned to any subagent.
- **C.** The report subagent's template lacks sections for policy and infrastructure; add those headings and the content will appear.
- **D.** The synthesis subagent dropped subtopics while condensing; increase its max_tokens so it can retain more of the search results.
<!--ANSWER
Correct: B
Tag: D1.2 · Coordinator decomposition scopes the whole problem
Why: The downstream agents worked correctly on what they were given, and cannot cover a subtopic that was never delegated. When the final output misses entire facets of a multi-facet question, the failure is upstream in how the coordinator decomposed the task, not in search quality, synthesis budget, or the report template. Fix the decomposition to enumerate the distinct dimensions (technology, infrastructure, policy, economics).
-->


In [ ]:
ex.answer(Q12="")


---

**Q13.** A coordinator should launch the search subagent and the document-analysis subagent to run at the same time, each needing the user's specific research question ("regulatory barriers to offshore wind in Japan, 2023-2025") to do useful work. In testing, the subagents run one after another and both return generic results about wind energy worldwide. Which TWO changes make them run concurrently AND on-target?  _(Select 2.)_

- **A.** Set a shared global variable holding the question and have each subagent read it at startup.
- **B.** Emit both Task calls in a single assistant response so the SDK dispatches them in parallel, rather than one Task call per turn.
- **C.** Rely on the subagents inheriting the coordinator's conversation history, which already contains the specific question, so no extra prompt text is needed.
- **D.** Increase the coordinator's temperature so it is more likely to issue two tool calls.
- **E.** Include the full, specific research question (topic, scope, date range) inside each Task prompt, because subagents start with isolated context and see only what the prompt passes.
<!--ANSWER
Correct: B, E
Tag: D1.3 · Parallel Task calls + explicit context in the prompt
Why: Subagents launched by the Task tool have isolated context — they do not inherit the coordinator's history, so the specific question must be written into each subagent's prompt (E); that is why both drifted to generic output. Concurrency comes from issuing multiple Task calls in one response (B); serial turns run them sequentially. History inheritance (C) and shared globals (A) are not how SDK subagent context works, and temperature (D) does not control parallelism.
-->


In [ ]:
ex.answer(Q13="")   # Select 2 — TWO letters


---

**Q14.** The report-generation subagent's only job is to turn already-synthesized, already-cited findings into formatted Markdown. During setup, an engineer grants it the same allowedTools as the coordinator: web fetch, a code sandbox, file write/delete, and the Task tool. What is the best correction?  _(Select 1.)_

- **A.** Scope the report subagent to only the formatting/output tools it actually needs, removing web fetch, the sandbox, delete, and especially Task, following least privilege.
- **B.** Give the report subagent the Task tool only, so it can delegate anything unexpected back out to fresh subagents.
- **C.** Keep the full tool set — the report agent may occasionally need to re-fetch a source, and having the tools available adds no cost until used.
- **D.** Move the report subagent onto a larger model so it can safely manage the broader tool set.
<!--ANSWER
Correct: A
Tag: D2.3 · Least-privilege, scoped tool distribution
Why: Each subagent should hold only the tools its role requires; a formatter needs output tools, not fetch, code execution, deletion, or the ability to spawn more subagents. Granting Task in particular lets a leaf agent recursively launch work it should never initiate. Unused-but-available tools expand the failure and misuse surface, so "no cost until used" (C) is wrong, and a bigger model (D) does not address privilege.
-->


In [ ]:
ex.answer(Q14="")


---

**Q15.** The document-analysis subagent is asked to extract merger-clause details from a set of PDFs. One PDF is password-protected and the tool call fails; the subagent returns `"No relevant information found."` The coordinator treats this identically to genuinely empty documents, reports the clause as absent, and moves on. Which TWO design changes let the coordinator handle this correctly?  _(Select 2.)_

- **A.** Have the subagent return a structured error object with a failure type (e.g. `access_denied`), the document it attempted, and any partial results, so the coordinator can distinguish an access failure from a real empty result.
- **B.** Suppress the failure and always return the friendliest available string so the workflow never stalls on one bad document.
- **C.** Abort the entire research run whenever any single document cannot be opened, to guarantee the report is never incomplete.
- **D.** Lower the analysis subagent's temperature so it stops hallucinating the "not found" message.
- **E.** Have the coordinator retry access-failure cases (or route them for credentials/alternative sources) while accepting genuine empty results as final, using the failure type to decide.
<!--ANSWER
Correct: A, E
Tag: D5.3 · Structured error propagation vs. valid-empty
Why: A subagent must surface failures as structured context — failure type, attempted target, partial results, alternatives — so the coordinator can tell an access failure (retryable/recoverable) apart from a valid empty result (final) and act accordingly (A, E). Collapsing an error into a success string (B) is exactly the bug that made the clause look absent, and killing the whole workflow on one unreadable file (C) is the opposite over-reaction. Temperature (D) is irrelevant; the string came from a swallowed tool error.
-->


In [ ]:
ex.answer(Q15="")   # Select 2 — TWO letters


---

**Q16.** The synthesis subagent receives two well-sourced figures for a company's FY2023 revenue: $4.1B from a Q4 earnings press release dated Jan 2024, and $3.9B from a market-research brief dated Aug 2023. It silently emits "Revenue was $4.1B" and discards the other value. Reviewers flag the report as unreliable. What should synthesis do instead?  _(Select 1.)_

- **A.** Escalate to a larger synthesis model and re-run, trusting it to reconcile the numbers into one correct figure.
- **B.** Keep only the value from the source it rates most authoritative and drop the other to avoid confusing the reader.
- **C.** Average the two figures to $4.0B so the report reflects both sources without contradiction.
- **D.** Report both values, annotated with their source and publication date, so the reader sees the discrepancy is a preliminary-vs-final timing difference rather than an error.
<!--ANSWER
Correct: D
Tag: D5.6 · Provenance & annotating conflicting sources
Why: When sources conflict, synthesis must preserve claim-to-source mappings and present both values with their provenance — including publication dates, which here reveal a preliminary-brief vs. finalized-earnings timing gap, not a genuine contradiction. Silently picking one (B), averaging unrelated figures (C), or asking a bigger model to invent a single reconciled number (A) all destroy the provenance the reader needs to judge the discrepancy.
-->


In [ ]:
ex.answer(Q16="")


---

**Q17.** The synthesis subagent hands off to the report subagent, which runs on a small context budget. Synthesis passes its full chain-of-thought — paragraphs of deliberation with source URLs, document names, and dates woven inline into the prose. The report agent frequently truncates, and citations end up attached to the wrong claims. What is the best fix?  _(Select 1.)_

- **A.** Pass synthesis's complete reasoning transcript but raise the report agent's model size so it can absorb the whole thing.
- **B.** Cache the full reasoning chain and have the report agent fetch pieces on demand as it writes each section.
- **C.** Hand off structured records that separate content (each finalized claim) from its metadata (source URL, document name, date), so the small-budget report agent gets compact facts with citations bound to the right claims.
- **D.** Strip all source URLs and dates before handoff so the prose fits the budget, and add citations back manually later.
<!--ANSWER
Correct: C
Tag: D5.6 · Structured context passing to small-budget agents
Why: A downstream agent with a small budget needs compact, structured facts — content separated from metadata (URLs, doc names, dates) — not a verbose reasoning transcript, which both blows the budget and lets citations drift off their claims. Structuring the handoff keeps each citation bound to its claim. Upsizing the model (A) is the over-engineered trap, stripping provenance (D) destroys the citations, and speculative caching/on-demand fetch (B) adds complexity without fixing the malformed handoff.
-->


In [ ]:
ex.answer(Q17="")


## Scenario 4 — Developer Productivity with Claude

_Primary domains: D2 Tool Design & MCP Integration · D3 Claude Code Configuration & Workflows · D1 Agentic Architecture & Orchestration._


---

**Q18.** An SDK-based agent that helps engineers navigate a 400k-line legacy monorepo is slow and often wrong on its first answer. Its system prompt instructs it to "Read the relevant modules, then answer." For a question like "where is the retry backoff for the payments client configured?" it opens a dozen large files, blows past the context window, and still misses the setting. What change to how the agent explores would most reliably fix this?  _(Select 1.)_

- **A.** Instruct it to Glob for `**/*payments*` and Read every matched file in full before answering, so it never misses the definition.
- **B.** Raise `max_tokens` and switch to a larger model so the whole set of candidate files fits in one context.
- **C.** Instruct it to Read the repository's top-level directories first to build a mental map, then Read each payments file end-to-end.
- **D.** Instruct it to Grep for `backoff`/`retry` to locate the defining file and line, then Read only that region to confirm the surrounding config — building understanding incrementally from search hits.
<!--ANSWER
Correct: D
Tag: D2.5 · Grep-first incremental exploration over read-everything
Why: Grep searches file *contents*, so it lands directly on the line where `backoff` is defined regardless of which file holds it — the exact "find a symbol/setting" job. You then Read only that region, keeping context small and the answer grounded. A and C are the read-everything anti-pattern that caused the context blowout; B treats a discovery-strategy problem as a capacity problem.
-->


In [ ]:
ex.answer(Q18="")


---

**Q19.** Your team's productivity agent integrates a company GitHub Enterprise MCP server. Two problems: (1) teammates who clone the repo don't get the server, they each hand-configure it; and (2) the config file that *does* work has the personal access token pasted in as a literal string, and it was nearly committed. Which two changes fix these while keeping secrets out of version control?  _(Select 2.)_

- **A.** Commit the token but rotate it weekly so a leaked value expires quickly.
- **B.** Define the server in each developer's `~/.claude.json` and email the file around so everyone has an identical copy.
- **C.** Move the token into a comment at the top of `.mcp.json` where scanners are less likely to flag it.
- **D.** Define the server in the project's `.mcp.json` at the repo root so it's shared with everyone who clones, and commit it.
- **E.** Reference the token as `${GITHUB_TOKEN}` in the config and have each developer set that variable in their own environment, so the committed file carries no secret.
<!--ANSWER
Correct: D, E
Tag: D2.4 · Shared .mcp.json + ${ENV} expansion for secrets
Why: A project `.mcp.json` is the shared, version-controlled home for team tooling, so a clone brings the server with it — fixing the per-developer setup drift. `${ENV_VAR}` expansion lets the committed config point at a secret that lives only in each person's environment, so nothing sensitive is checked in. B recreates the drift by copying personal config; A and C still commit the secret.
-->


In [ ]:
ex.answer(Q19="")   # Select 2 — TWO letters


---

**Q20.** An engineer asks the agent to "rename the `Invoice.total` field to `grandTotal` across the billing service." The agent tries `Edit` on `invoice.py` with the old text `total` and the new text `grandTotal`, and the call fails because `total` appears eleven times in the file. What is the correct way to think about this, and the right recovery?  _(Select 1.)_

- **A.** `Edit` only works on files created in the current session; recreate `invoice.py` with Write, then Edit will match.
- **B.** Use Glob to find every file named `invoice.py` and Write each one fresh — Edit is not designed for renames.
- **C.** `Edit` failed because the file is too large; split `invoice.py` into smaller files first, then Edit each.
- **D.** `Edit` requires a unique text anchor; when no unique match exists, Grep for the occurrences to see them in context, then use targeted Edits with larger surrounding snippets that *are* unique — or Read the file and Write it back with the change.
<!--ANSWER
Correct: D
Tag: D2.5 · Edit needs a unique anchor; fall back to wider anchors or Read+Write
Why: `Edit` performs a targeted change keyed to a *unique* text match; a token appearing eleven times is ambiguous, which is exactly why it errored. The fix is to make each anchor unique (more surrounding context) or, when that's impractical, Read the whole file and Write it back. C, A, and B invent constraints Edit doesn't have and reach for the wrong tools.
-->


In [ ]:
ex.answer(Q20="")


---

**Q21.** The agent is asked to "add OAuth login to this Flask app you've never seen before." On its first attempt it immediately starts editing `app.py`, adds a route that collides with an existing one, and installs a library the project already vendors a different version of. You want a more reliable outcome without babysitting every step. Which approach best fits?  _(Select 1.)_

- **A.** Lower the model's temperature so it makes fewer mistakes on the first edit.
- **B.** Tell it to skip inspection and just write the OAuth code from the framework's canonical tutorial, since Flask OAuth is standard.
- **C.** Have it produce a plan first — inspect the existing auth/routing and dependency setup, then propose the concrete edits for review — before it touches any files.
- **D.** Give it broader permissions (auto-approve Write and Bash) so it can iterate faster through its mistakes.
<!--ANSWER
Correct: C
Tag: D3.4 · Plan before direct execution on unfamiliar, high-uncertainty work
Why: The failures (route collision, dependency clash) come from acting before understanding an unfamiliar codebase — the signature case for planning first: gather context, propose edits, get them reviewed, then execute. D amplifies the damage by auto-approving destructive tools; B ignores the project's actual state that caused the collisions; A tweaks a knob that doesn't address missing context.
-->


In [ ]:
ex.answer(Q21="")


---

**Q22.** During a long refactor session (`--resume feature-auth`), the agent had a solid mental model of the module. While it was idle you pulled `main`, which renamed `auth/session.py` to `auth/tokens.py` and changed several signatures. You resume the same session and the agent keeps referencing the old path and stale function shapes, producing broken edits. What is the best way to proceed?  _(Select 1.)_

- **A.** Use `fork_session` to branch the stale session, since a fork automatically re-scans the filesystem and discards outdated file contents.
- **B.** Tell the resumed session explicitly what changed on disk (the rename and the new signatures) so it re-reads the current files; if its working context is now largely stale, start a fresh session seeded with a short summary of the goal instead.
- **C.** Keep resuming the same session and re-run the edits — the agent will eventually reconcile its memory with the files on its own.
- **D.** Delete the session history file so the next resume is forced to rebuild everything from scratch, then resume by the same name.
<!--ANSWER
Correct: B
Tag: D1.7 · Inform resumed sessions of file changes; fresh+summary beats stale resume
Why: A resumed session carries its prior context, including now-outdated file contents, so it must be *told* what changed on disk to re-read the current state. When too much of that context is stale, a fresh session seeded with a concise summary is more reliable than fighting a resume. C assumes self-correction that won't happen; forking (A) copies the stale context rather than refreshing it; deleting history (D) isn't how resume-by-name works and needlessly discards the goal context.
-->


In [ ]:
ex.answer(Q22="")


## Scenario 5 — Claude Code for Continuous Integration

_Primary domains: D3 Claude Code Configuration & Workflows · D4 Prompt Engineering & Structured Output._


---

**Q23.** A team's CI job runs `claude -p "Review the diff and list every bug"` on each pull request. The pipeline captures Claude's stdout and tries to post each finding as an inline comment on the exact file and line, but the parser breaks constantly because Claude returns prose paragraphs whose formatting drifts from run to run. What is the most reliable fix?  _(Select 1.)_

- **A.** Lower the model's temperature and append "always use the same Markdown headings" so the prose format stays stable enough to regex.
- **B.** Set the `CLAUDE_HEADLESS=1` environment variable so Claude Code emits terminal-free plain output the parser can split on newlines.
- **C.** Switch the job to interactive mode and scrape the rendered TUI, which produces a consistent visual layout.
- **D.** Add `--output-format json` and pass a `--json-schema` that requires a `findings` array of `{file, line, severity, message}` objects.
<!--ANSWER
Correct: D
Tag: D3.6 · headless structured output for CI
Why: `--output-format json` combined with a `--json-schema` forces machine-parseable, schema-validated findings that map cleanly onto inline PR comments — the only robust option. `CLAUDE_HEADLESS` is not a real variable, and coaxing stable prose (A) or scraping the TUI (C) both keep the fragile free-text parsing that caused the failure.
-->


In [ ]:
ex.answer(Q23="")


---

**Q24.** To cut cost, a team moved its blocking pre-merge review — the check that must pass before a PR can merge — onto the Message Batches API, since batching is ~50% cheaper. Now PRs sit for hours waiting on review results, blocking developers. What is the correct architecture?  _(Select 1.)_

- **A.** Keep everything on Batches but poll every 30 seconds and cancel-and-resubmit any request that has not completed in two minutes.
- **B.** Keep the blocking review on synchronous Messages API calls for immediate results, and reserve the Batches API for the non-blocking nightly full-repo audit.
- **C.** Stay on Batches but add `--batch --priority high` so blocking jobs jump ahead of the queue and return within the SLA.
- **D.** Split each PR into one batch request per file so smaller requests complete faster and unblock the merge sooner.
<!--ANSWER
Correct: B
Tag: D4.5 · batch vs synchronous
Why: The Batches API offers no latency guarantee (up to 24h) and is designed for large asynchronous jobs, making it wrong for a merge-blocking check; synchronous calls are the right tool there, with batch kept for the overnight audit. There is no `--batch`/`--priority` flag or SLA (C), and polling or per-file splitting (A, D) cannot manufacture a latency guarantee the API does not provide.
-->


In [ ]:
ex.answer(Q24="")


---

**Q25.** A CI stage uses a single long-lived Claude Code session to (1) generate new test cases for a changed module and then (2) review those same generated tests for correctness and coverage gaps. The generated tests compile and pass, but reviewers keep finding logic bugs the review step should have caught. What change most improves bug detection?  _(Select 1.)_

- **A.** Run the review in a second, independent Claude instance that did not generate the tests, giving it only the tests and the module under test.
- **B.** Ask the same session to "critique your own tests harshly and assume they contain at least three bugs" before finishing.
- **C.** Increase the review step's `max_tokens` and move to a larger-context model so the session can hold more of the code at once.
- **D.** Have the session regenerate the tests three times and keep whichever version it rates highest.
<!--ANSWER
Correct: A
Tag: D4.6 · independent review instance
Why: A session that produced the code carries its own reasoning and blind spots forward, so it is systematically worse at auditing its own output; a fresh, independent instance evaluates without that prior commitment. More tokens/context (C) does not address the bias, and self-critique or self-ranking (B, D) still runs inside the same compromised session.
-->


In [ ]:
ex.answer(Q25="")


---

**Q26.** An automated PR reviewer is technically accurate but developers have started ignoring it: most of its comments are subjective style preferences ("prefer a guard clause here", "this name could be clearer"), burying the occasional real security finding. Its prompt currently says "be conservative and only comment when confident." Which TWO changes best restore trust?  _(Select 2.)_

- **A.** Replace "be conservative" with explicit categorical criteria: report correctness bugs, security issues, and data-loss risks; skip formatting and naming preferences.
- **B.** Add a final instruction to summarize all minor style issues into one collapsed comment so nothing is lost but the noise is grouped.
- **C.** Raise the confidence bar further by instructing the model to comment only when it is "extremely certain and the issue is severe."
- **D.** Temporarily disable the naming/style category while its prompt is rewritten, so its false positives stop reaching developers during the fix.
- **E.** Route every comment through a second model that rates each on a 1–10 helpfulness scale and drops those below 7.
<!--ANSWER
Correct: A, D
Tag: D4.1 · explicit categorical criteria / false-positive control
Why: Vague guidance like "be conservative" or "extremely certain" (C) gives the model no shared definition of what matters, so explicit report-vs-skip categories (A) are what actually cut the noise. Because false positives erode trust immediately, temporarily disabling the offending category while its prompt is fixed (D) stops the damage now. Collapsing (B) or a numeric-score gate (E) keep generating the same low-value comments rather than removing the category driving them away.
-->


In [ ]:
ex.answer(Q26="")   # Select 2 — TWO letters


---

**Q27.** A PR review runs on every push. When a developer pushes follow-up commits addressing feedback, Claude reviews the full updated diff again and re-posts comments for issues it already flagged (some now fixed), cluttering the thread with duplicates. What is the best way to prevent duplicate comments?  _(Select 1.)_

- **A.** Include the prior review's findings in the prompt and instruct Claude to report only issues that are new or still unaddressed in the latest commits.
- **B.** Add a `custom_id` to each finding so the PR platform automatically deduplicates identical comments.
- **C.** Only review the incremental diff of the newest commit so previously reviewed lines are never revisited.
- **D.** Delete all of Claude's earlier comments before each run so the thread only ever shows the latest full review.
<!--ANSWER
Correct: A
Tag: D3.6 · dedupe on re-run
Why: Feeding the previous findings back as context and instructing the model to surface only new or unresolved issues is what lets it reason about what has already been said and what a fix changed. Reviewing only the newest commit (C) misses issues whose fix spans earlier lines; wiping prior comments (D) destroys the review history; and `custom_id` (B) correlates batch requests to responses — it is not a PR-comment dedup mechanism.
-->


In [ ]:
ex.answer(Q27="")


## Scenario 6 — Structured Data Extraction

_Primary domains: D4 Prompt Engineering & Structured Output · D5 Context Management & Reliability._


---

**Q28.** An extraction system pulls invoice fields into a JSON schema. Every response now conforms to the schema — no more parse failures — yet downstream accounting still rejects ~8% of records because `line_items` sums don't equal the `invoice_total`, and occasionally the tax amount lands in the `subtotal` field. The team wants to know why schema enforcement didn't fix these. What is the correct explanation and next step?  _(Select 1.)_

- **A.** Schema validation ran but the model's temperature is too high; lower it to 0 and the totals will always match.
- **B.** JSON Schema supports arithmetic constraints natively; the schema simply omits a `sum`-of relationship, so declare it and the API will reject non-summing outputs.
- **C.** tool_use with a JSON schema guarantees only that output is structurally valid; it cannot enforce semantic correctness like arithmetic consistency or correct field assignment, so add a self-correction step that extracts `calculated_total` alongside the stated total and flags mismatches.
- **D.** The schema is being applied with `tool_choice: "auto"`, so the model sometimes returns prose; switch to `"any"` and the sums will reconcile.
<!--ANSWER
Correct: C
Tag: D4.3 · structured output eliminates syntax errors, not semantic errors
Why: Forcing a tool call with a JSON schema guarantees schema-valid *shape*, but the model can still put values in the wrong field or produce line items that don't sum — those are semantic errors. The fix is a self-correction/verification flow (extract a `calculated_total` and compare to the stated total, flag discrepancies). tool_choice and temperature don't address semantics, and JSON Schema has no cross-field arithmetic constraints.
-->


In [ ]:
ex.answer(Q28="")


---

**Q29.** A contract-extraction schema marks `governing_law` and `termination_notice_days` as **required** strings. On contracts that simply don't state a governing-law clause, the model confidently invents a plausible jurisdiction rather than leaving it out. Which schema changes reduce this fabrication? _(Select 2.)_

- **A.** Set `tool_choice` to a forced `{"type":"tool","name":"extract_contract"}` so the model must call the tool every time.
- **B.** Make `governing_law` nullable/optional so the model can omit it when the source lacks the clause, instead of being forced to satisfy a required field.
- **C.** Add an enum value like `"not_specified"` (plus an optional `detail` field) so "absent" is a first-class, representable outcome.
- **D.** Increase `max_tokens` so the model has room to reason before emitting the field.
<!--ANSWER
Correct: B, C
Tag: D4.3 · nullable/optional + enum sentinel prevent fabrication of required fields
Why: A required field forces the model to produce *something*, which invites fabrication when the source is silent. Making the field nullable/optional, and adding an explicit sentinel enum ("not_specified") with an optional detail, gives the model a truthful way to represent absence. Forcing the tool call and raising max_tokens change when/how much the model outputs, not whether it must fabricate a required value.
-->


In [ ]:
ex.answer(Q29="")   # Select 2 — TWO letters


---

**Q30.** A validation layer checks each extraction and, on failure, retries by re-sending the document, the failed extraction, and the specific error. It works well for malformed dates and out-of-range enums, but for one failure class it loops the full 3 retries and still fails every time, wasting tokens. Which failure class is retry-with-feedback fundamentally unable to fix, and why? _(Select 1.)_

- **A.** A `currency` value of `"US Dollars"` that fails an enum expecting `"USD"` — the model can map it on retry.
- **B.** A `line_items` array the model truncated mid-object because it hit the token limit — resending with the error lets it complete the structure.
- **C.** A date returned as `03/04/2025` that violates the ISO-8601 format constraint — the error feedback tells the model exactly how to reformat.
- **D.** A required `purchase_order_number` that is genuinely not present anywhere in the source document — retry cannot recover information that does not exist in the input.
<!--ANSWER
Correct: D
Tag: D4.4 · retry is useless when the info is absent (vs format/structural errors)
Why: Validation-plus-retry with error feedback fixes *format and structural* problems (bad date format, wrong enum spelling, truncated JSON) because the needed information exists and only its representation is wrong. When a required value is simply absent from the source, no amount of re-prompting can conjure it — the correct design routes these to human review or a nullable field rather than retrying.
-->


In [ ]:
ex.answer(Q30="")


---

**Q31.** The extraction system reports **97% aggregate field accuracy** on a held-out set, so leadership wants to auto-approve everything and drop human review. An architect objects. Which practices correctly address the risk the aggregate number hides? _(Select 2.)_

- **A.** Use stratified sampling by document type and field, because a high overall average can mask that a rare-but-critical doc type (e.g., handwritten remittances) or a specific field is well below 97%.
- **B.** Raise the aggregate target to 99% across the whole set; once the single number is high enough, per-type gaps no longer matter.
- **C.** Attach per-field calibrated confidence scores and route only low-confidence extractions to human reviewers, rather than trusting one global accuracy figure.
- **D.** Replace the human-review queue with a second Claude call that re-scores the first extraction and auto-approves anything it agrees with.
<!--ANSWER
Correct: A, C
Tag: D5.5 · aggregate accuracy hides per-type/per-field gaps; stratify + calibrated confidence
Why: A single aggregate accuracy figure can hide that specific document types or fields perform far worse than average. Stratified sampling surfaces those pockets, and per-field *calibrated* confidence lets you route genuinely uncertain extractions to humans. A second model auto-approving its own kind of errors isn't independent human oversight, and pushing the aggregate higher doesn't reveal or fix the per-type gaps.
-->


In [ ]:
ex.answer(Q31="")   # Select 2 — TWO letters


---

**Q32.** A long-running extraction agent processes a 90-page deposition in chunks, keeping a running *progressive summary* of prior chunks as context. Late-document extractions of dollar amounts and dates have started drifting from the source — a $1,450,000 settlement figure from page 6 comes back as "approximately $1.5 million." What is the best design fix? _(Select 1.)_

- **A.** Move the running summary to the very middle of the context window so the model attends to it most.
- **B.** Preserve exact numeric case-facts (amounts, dates, IDs) verbatim in a structured, non-summarized store that persists across chunks, rather than letting them be paraphrased into the progressive summary.
- **C.** Increase the summary length so more of each chunk survives compression.
- **D.** Lower temperature to 0 so the model stops rounding numbers.
<!--ANSWER
Correct: B
Tag: D5.1 · preserve exact numeric facts outside progressive summaries
Why: Progressive summarization is lossy and paraphrases precise values ("$1,450,000" -> "about $1.5M"). The reliable fix is to keep exact numeric case-facts verbatim in a structured store that isn't subject to summarization, so they remain retrievable regardless of chunk position. A longer summary still paraphrases; the middle of the window is actually where attention is weakest (lost-in-the-middle); temperature doesn't restore a value already lost to summarization.
-->


In [ ]:
ex.answer(Q32="")


---

**Q33.** A research-paper extractor must pull citations, but papers arrive in two very different formats: some use inline numeric citations `[12]` with a bibliography, others use inline author-year `(Smith, 2019)`. A zero-shot prompt returns empty `citations` arrays for whichever format it "didn't expect," and occasionally hallucinates entries. Which prompt-engineering change most directly improves extraction across both formats? _(Select 1.)_

- **A.** Provide few-shot examples covering *both* citation styles, showing the correct extraction for an inline-numeric+bibliography paper and for an author-year paper.
- **B.** Batch the papers through the Message Batches API to cut cost by 50%.
- **C.** Add a system prompt line saying "be accurate and never miss a citation."
- **D.** Switch `tool_choice` to `"any"` so the model is forced to call some tool on every paper.
<!--ANSWER
Correct: A
Tag: D4.2 · few-shot examples for varied document structures reduce empty/hallucinated extraction
Why: Empty arrays for one format and hallucinations for the other are symptoms of the model not having seen the structural variety it must handle. Few-shot examples that demonstrate correct extraction for each citation style teach the model to recognize and extract both, reducing null and fabricated outputs. Forcing a tool call doesn't teach format handling, vague exhortations don't change behavior reliably, and batching only affects cost/throughput, not accuracy.
-->


In [ ]:
ex.answer(Q33="")


In [ ]:
# ✅ Anything left blank? (run this before grading)
ex.remaining()


In [ ]:
# 🎯 GRADE — score, breakdowns by scenario/domain/type, and the rationale
# for EVERY miss. Writes the report to personal/ (git-ignored).
ex.grade()


---

## After grading

`ex.grade()` already did the mechanical part: score, breakdowns by scenario / domain / type,
and the `Why:` for each miss. What's left is yours:

- **Diagnose by domain, not by question.** One stray miss is noise; three in the same domain is
  a gap. Weights decide priority: **D1 27% · D3 20% · D4 20%**.
- **Name the distractor.** For each miss, say out loud which pattern caught you — over-engineering,
  proxy metric, nonexistent feature, prompt-where-determinism-was-needed, wrong component.
  See [`../DISTRACTOR_HEURISTIC.md`](../DISTRACTOR_HEURISTIC.md).
- **Re-derive, don't memorize.** State the *principle* in one line ("guaranteed requirement ->
  deterministic gate, not a prompt") before moving on. That judgment is what the exam tests; the
  correct letter buys you nothing tomorrow.
- **Locate the Task Statement** for each miss in [`../MAPPING.md`](../MAPPING.md) and re-read that
  section of the domain notebook.

The report is saved under `personal/` (git-ignored) so you can compare a second attempt against it.
